<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp?1" width="100px"></a>
</td>
</tr>
</table>


# 第 7 章：微调以遵循指令


In [ ]:
from importlib.metadata import version

pkgs = [
    "numpy",       # PyTorch 与 TensorFlow 依赖
    "matplotlib",  # 绘图库
    "tiktoken",    # 分词器
    "torch",       # 深度学习库
    "tqdm",        # 进度条
    "tensorflow",  # 用于 OpenAI 预训练权重
]
for p in pkgs:
    print(f"{p} version: {version(p)}")


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/01.webp" width=500px>

&nbsp;
## 7.1 指令微调简介


- 在第 5 章中，我们看到预训练 LLM 涉及一种训练过程，模型逐词学习生成文本
- 因此，预训练 LLM 擅长文本补全，但不擅长遵循指令
- 在本章中，我们将教会 LLM 更好地遵循指令


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/02.webp" width=500px>

- 本章涵盖的主题总结如下图所示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/03.webp" width=500px>


&nbsp;
## 7.2 准备用于监督指令微调的数据集


- 我们将使用我为本章准备的一个指令数据集


In [ ]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


# 本书最初使用以下代码
# 然而，urllib 使用较旧的协议设置，
# 可能导致部分使用 VPN 的读者出现问题。
# 上面的 `requests` 版本在这方面更稳健。

"""
import urllib

def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data
"""


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("条目数量:", len(data))


- 上面从 JSON 文件加载的 `data` 列表中，每个元素都是如下形式的字典


In [ ]:
print("示例条目:\n", data[50])


- 注意 `'input'` 字段可以为空：


In [ ]:
print("另一个示例条目:\n", data[999])


- 指令微调通常被称为"监督指令微调"，因为它在输入-输出对明确提供的数据集上训练模型
- 将条目格式化为 LLM 输入的方式有多种；下图展示了用于训练 Alpaca（https://crfm.stanford.edu/2023/03/13/alpaca.html）和 Phi-3（https://arxiv.org/abs/2404.14219）LLM 的两种示例格式


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/04.webp?2" width=500px>

- 在本章中，我们使用 Alpaca 风格的提示格式，这是指令微调的原始提示模板
- 下面，我们格式化将作为 LLM 输入传入的内容


In [ ]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- 带 input 字段的格式化响应如下所示


In [ ]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

- 下面是不带 input 字段的格式化响应


In [ ]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

- 最后，在下一节准备 PyTorch 数据加载器之前，我们将数据集划分为训练集、验证集和测试集


In [ ]:
train_portion = int(len(data) * 0.85)  # 85% 用于训练
test_portion = int(len(data) * 0.1)    # 10% 用于测试
val_portion = len(data) - train_portion - test_portion  # 剩余 5% 用于验证

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]


In [ ]:
print("训练集长度:", len(train_data))
print("验证集长度:", len(val_data))
print("测试集长度:", len(test_data))


&nbsp;
## 7.3 将数据组织成训练批次


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/05.webp?1" width=500px>

- 我们分几个步骤处理数据集批处理，如下图所示

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/06.webp?1" width=500px>


- 首先，我们实现一个 `InstructionDataset` 类，对数据集中所有输入进行预分词，类似于第 6 章中的 `SpamDataset`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/07.webp?1" width=500px>


In [ ]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # 预分词文本
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


- 与第 6 章类似，我们希望在批次中收集多个训练样本以加速训练；这要求将所有输入填充到相似的长度
- 同样与上一章类似，我们使用 `<|endoftext|>` token 作为填充 token


In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

- 在第 6 章中，我们将数据集中所有样本填充到相同长度
  - 这里，我们采用更复杂的方法，开发一个自定义 "collate" 函数，可以传递给数据加载器
  - 该自定义 collate 函数将每个批次中的训练样本填充到相同长度（但不同批次可以有不同的长度）


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/08.webp?1" width=500px>

In [ ]:
def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 找到批次中最长的序列
    # 并将最大长度 +1，这将在下面添加一个额外的
    # 填充 token
    batch_max_length = max(len(item)+1 for item in batch)

    # 填充并准备输入
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        # 添加一个 <|endoftext|> token
        new_item += [pad_token_id]
        # 将序列填充到 batch_max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        # 通过 padded[:-1]，我们移除通过 batch_max_length 中 +1 设置
        # 添加的额外填充 token
        # （额外的填充 token 将在后续代码中用到）
        inputs = torch.tensor(padded[:-1])
        inputs_lst.append(inputs)

    # 将输入列表转换为张量并传输到目标设备
    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor


In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/09.webp?1" width=400px>

- 上面，我们只返回了 LLM 的输入；然而，对于 LLM 训练，我们还需要目标值
- 与预训练 LLM 类似，目标是输入向右移动 1 个位置，这样 LLM 学习预测下一个 token


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/10.webp?1" width=400px>

In [ ]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    # 找到批次中最长的序列
    batch_max_length = max(len(item)+1 for item in batch)

    # 填充并准备输入
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # 添加一个 <|endoftext|> token
        new_item += [pad_token_id]
        # 将序列填充到 max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 截断最后一个 token 作为输入
        targets = torch.tensor(padded[1:])  # 右移 +1 作为目标
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 将输入列表转换为张量并传输到目标设备
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor


In [ ]:
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

- 接下来，我们引入 `ignore_index` 值，用新值替换所有填充 token ID；`ignore_index` 的目的是我们可以在损失函数中忽略填充值（稍后会详细介绍）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/11.webp?1" width=400px>

- 具体来说，这意味着我们将对应于 `50256` 的 token ID 替换为 `-100`，如下图所示


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/12.webp?2" width=500px>

- （此外，我们还引入 `allowed_max_length`，以防我们想限制样本长度；如果您计划使用超过 GPT-2 模型支持的 1024 token 上下文长度的自定义数据集，这将很有用）


In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # 找到批次中最长的序列
    batch_max_length = max(len(item)+1 for item in batch)

    # 填充并准备输入和目标
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        # 添加一个 <|endoftext|> token
        new_item += [pad_token_id]
        # 将序列填充到 max_length
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  # 截断最后一个 token 作为输入
        targets = torch.tensor(padded[1:])  # 右移 +1 作为目标

        # 新增：将目标中除第一个外的所有填充 token 替换为 ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # 新增：可选地截断到最大序列长度
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # 将输入和目标列表转换为张量并传输到目标设备
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor


In [ ]:
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

- 让我们看看用 -100 替换实现了什么
- 为便于说明，假设我们有一个类似第 6 章的小型分类任务，有 2 个类别标签 0 和 1
- 如果我们有以下 logits 值（模型最后一层的输出），我们计算如下损失


In [ ]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  # 第 1 个训练样本
     [-0.5, 1.5]]  # 第 2 个训练样本
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)


- 现在，添加一个训练样本会如预期地影响损失


In [ ]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # 新增的第 3 个训练样本
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)


- 让我们看看如果将其中一个样本的类别标签替换为 -100 会发生什么


In [ ]:
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)


- 如我们所见，这 3 个训练样本的损失与从 2 个训练样本计算的损失相同，这意味着交叉熵损失函数忽略了标签为 -100 的训练样本
- 默认情况下，PyTorch 的 `cross_entropy(..., ignore_index=-100)` 设置会忽略对应标签 -100 的样本
- 使用这个 -100 `ignore_index`，我们可以忽略批次中用于将训练样本填充到等长的额外 end-of-text（填充）token
- 然而，我们不想忽略 end-of-text（填充）token（50256）的第一个实例，因为它可以帮助 LLM 知道响应何时完成


- 在实践中，屏蔽掉对应于指令的目标 token ID 也很常见，如下图所示（这是完成本章后推荐的读者练习）


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/13.webp" width=600px>

&nbsp;
## 7.4 为指令数据集创建数据加载器


- 在本节中，我们使用 `InstructionDataset` 类和 `custom_collate_fn` 函数来实例化训练、验证和测试数据加载器


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/14.webp" width=500px>

- 之前 `custom_collate_fn` 函数的另一个额外细节是，我们现在直接将数据移动到目标设备（例如 GPU），而不是在主训练循环中执行，这提高了效率，因为当我们将 `custom_collate_fn` 作为数据加载器的一部分使用时，它可以作为后台进程执行
- 使用 Python `functools` 标准库中的 `partial` 函数，我们创建一个预填充了原始函数 `device` 参数的新函数


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # 使用 PyTorch 2.9 或更新版本以获得稳定的 mps 结果
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("设备:", device)


In [ ]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

- 接下来，我们实例化数据加载器，与前几章类似，只是现在为批处理过程提供我们自己的 collate 函数


In [ ]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [ ]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

- 让我们看看生成的输入和目标批次的维度


In [ ]:
print("训练加载器:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)


- 根据上面的输出，所有批次的 batch size 均为 8，但长度不同，符合预期
- 让我们再检查一下输入是否包含对应 token ID 50256 的 `<|endoftext|>` 填充 token，打印 `inputs` 批次中第一个训练样本的内容


In [ ]:
print(inputs[0])

- 同样，我们直观地检查目标是否包含 -100 占位符 token


In [ ]:
print(targets[0])

&nbsp;
## 7.5 加载预训练 LLM


- 在本节中，我们使用第 5 章 5.5 节和第 6 章 6.4 节中相同的代码加载预训练 GPT 模型


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/15.webp?1" width=500px>

- 然而，我们不加载最小的 1.24 亿参数模型，而是加载 3.55 亿参数的中等版本，因为 1.24 亿模型太小，无法通过指令微调获得质量合理的结果


In [ ]:
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# 如果本地没有 `previous_chapters.py` 文件，
# 可以从 `llms-from-scratch` PyPI 包导入。
# 详情见：https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# 例如：
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import download_and_load_gpt2, load_weights_into_gpt


BASE_CONFIG = {
    "vocab_size": 50257,     # 词表大小
    "context_length": 1024,  # 上下文长度
    "drop_rate": 0.0,        # Dropout 率
    "qkv_bias": True         # Query-key-value 偏置
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();


- 在下一节开始微调模型之前，让我们看看它在一个验证任务上的表现


In [ ]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
print(input_text)

In [ ]:
from previous_chapters import (
    generate,
    text_to_token_ids,
    token_ids_to_text
)
# 或者：
# from llms_from_scratch.ch05 import (
#    generate,
#    text_to_token_ids,
#    token_ids_to_text
# )


token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)


- 注意，我们在前几章使用的 `generate` 函数返回合并的输入和输出文本，这在上一节创建可读文本时很方便
- 要隔离响应，我们可以从 `generated_text` 开头减去指令的长度


In [ ]:
response_text = (
    generated_text[len(input_text):]
    .replace("### Response:", "")
    .strip()
)
print(response_text)

- 如我们所见，模型尚不能遵循指令；它创建了 "Response" 部分，但只是重复了原始输入句子和指令


&nbsp;
## 7.6 在指令数据上微调 LLM


- 在本节中，我们微调模型

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/16.webp" width=500px>

- 注意，我们可以复用前几章使用的所有损失计算和训练函数


In [ ]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)
# 或者：
# from llms_from_scratch.ch05 import (
#    calc_loss_loader,
#    train_model_simple,
# )




- 在开始训练之前，让我们计算初始的训练集和验证集损失（与前几章一样，目标是最小化损失）


In [ ]:
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("训练损失:", train_loss)
print("验证损失:", val_loss)


- 注意，训练比前几章更昂贵，因为我们使用的是更大的模型（3.55 亿参数而非 1.24 亿参数）
- 下方列出了各设备的运行时间供参考（在兼容 GPU 设备上运行此 notebook 无需修改代码）


<div style="text-align: left;">
    
| 模型 | 设备 | 2 个 Epoch 运行时间 |
|--------------------|-----------------------|----------------------|
| gpt2-medium (355M) | CPU (M3 MacBook Air)  | 15.78 分钟        |
| gpt2-medium (355M) | GPU (M3 MacBook Air)  | 10.77 分钟        |
| gpt2-medium (355M) | GPU (L4)              | 1.83 分钟         |
| gpt2-medium (355M) | GPU (A100)            | 0.86 分钟         |
| gpt2-small (124M)  | CPU (M3 MacBook Air)  | 5.74 分钟         |
| gpt2-small (124M)  | GPU (M3 MacBook Air)  | 3.73 分钟         |
| gpt2-small (124M)  | GPU (L4)              | 0.69 分钟         |
| gpt2-small (124M)  | GPU (A100)            | 0.39 分钟         |

</div>

- 我使用 `"gpt2-medium (355M)"` 模型运行了此 notebook


In [ ]:
import time

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"训练完成，耗时 {execution_time_minutes:.2f} 分钟。")


- 根据上面的输出，模型训练良好，可以从不断下降的训练损失和验证损失值看出
- 此外，根据每个 epoch 后打印的响应文本，我们可以看到模型正确遵循了将输入句子 `'The chef cooks the meal every day.'` 转换为被动语态 `'The meal is cooked every day by the chef.'` 的指令（我们将在后面的章节中正确格式化和评估响应）
- 最后，让我们看一下训练和验证损失曲线


In [ ]:
from previous_chapters import plot_losses
# 或者：
# from llms_from_scratch.ch05 import plot_losses

epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)


- 如我们所见，损失在第一个 epoch 开始时急剧下降，这意味着模型开始快速学习
- 我们可以看到，轻微过拟合在约 1 个训练 epoch 时出现


&nbsp;
## 7.7 提取并保存响应


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/18.webp?1" width=500px>

- 在本节中，我们保存测试集响应以便在下一节中评分
- 我们还保存一份模型副本以供将来使用
- 但首先，让我们简要看一下微调模型生成的响应


In [ ]:
torch.manual_seed(123)


for entry in test_data[:3]:

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
)

    print(input_text)
    print(f"\nCorrect response:\n>> {entry['output']}")
    print(f"\nModel response:\n>> {response_text.strip()}")
    print("-------------------------------------")


- 根据测试集指令、给定响应和模型响应，模型表现相对较好
- 第一个和最后一个指令的答案明显正确
- 第二个答案接近；模型回答 "cumulus cloud" 而非 "cumulonimbus"（然而，请注意 cumulus 云可以发展成 cumulonimbus 云，能够产生雷暴）
- 最重要的是，我们可以看到模型评估不像上一章那样简单，上一章只需计算正确 spam/非 spam 类别标签的百分比即可获得分类准确率
- 在实践中，指令微调 LLM（如聊天机器人）通过多种方法进行评估
  - 短答案和多选题基准，如 MMLU（"Measuring Massive Multitask Language Understanding"，[https://arxiv.org/abs/2009.03300](https://arxiv.org/abs/2009.03300)），测试模型的知识
  - 与其他 LLM 的人类偏好比较，如 LMSYS chatbot arena（[https://arena.lmsys.org](https://arena.lmsys.org)）
  - 自动化对话基准，其中另一个 LLM（如 GPT-4）用于评估响应，如 AlpacaEval（[https://tatsu-lab.github.io/alpaca_eval/](https://tatsu-lab.github.io/alpaca_eval/)）

- 在下一节中，我们将使用类似 AlpacaEval 的方法，使用另一个 LLM 评估我们模型的响应；但是，我们将使用自己的测试集，而不是使用公开可用的基准数据集
- 为此，我们将模型响应添加到 `test_data` 字典中，并将其保存为 `"instruction-data-with-response.json"` 文件以供记录，以便在需要时可以在单独的 Python 会话中加载和分析


In [ ]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)
    response_text = generated_text[len(input_text):].replace("### Response:", "").strip()

    test_data[i]["model_response"] = response_text


with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # "indent" 用于美化输出


- 让我们再次检查其中一个条目，确认响应已正确添加到 `test_data` 字典中


In [ ]:
print(test_data[0])

- 最后，我们还保存模型，以便将来可以复用


In [ ]:
import re


file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL) }-sft.pth"
torch.save(model.state_dict(), file_name)
print(f"模型已保存为 {file_name}")

# 通过以下方式加载模型
# model.load_state_dict(torch.load("gpt2-medium355M-sft.pth"))


&nbsp;
## 7.8 评估微调后的 LLM


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/19.webp?1" width=500px>

- 在本节中，我们使用另一个更大的 LLM 自动化评估微调 LLM 的响应
- 具体来说，我们使用 Meta AI 的指令微调 80 亿参数 Llama 3 模型，可以通过 ollama（[https://ollama.com](https://ollama.com)）在本地运行
- （或者，如果您更喜欢通过 OpenAI API 使用更强大的 LLM（如 GPT-4），请参阅 [llm-instruction-eval-openai_ch.ipynb](../03_model-evaluation/llm-instruction-eval-openai_ch.ipynb) notebook）


- Ollama 是一个高效运行 LLM 的应用程序
- 它是 llama.cpp（[https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)）的封装，llama.cpp 用纯 C/C++ 实现 LLM 以最大化效率
- 注意，它是用于使用 LLM 生成文本（推理）的工具，而非训练或微调 LLM
- 在运行下面的代码之前，请访问 [https://ollama.com](https://ollama.com) 并按照说明安装 ollama（例如，点击 "Download" 按钮并下载适用于您操作系统的 ollama 应用程序）


- 对于 macOS 和 Windows 用户，点击您下载的 ollama 应用程序；如果提示安装命令行用法，请选择 "yes"
- Linux 用户可以使用 ollama 网站上提供的安装命令

- 一般来说，在从命令行使用 ollama 之前，我们必须启动 ollama 应用程序或在单独的终端中运行 `ollama serve`

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/20.webp?1" width=700px>


---

**注意**：

- 如上所述，在终端中运行 `ollama serve` 时，您可能会遇到错误消息 `Error: listen tcp 127.0.0.1:11434: bind: address already in use`
- 如果是这种情况，请尝试使用命令 `OLLAMA_HOST=127.0.0.1:11435 ollama serve`（如果此地址也被占用，请尝试将数字递增一，直到找到未使用的地址）

---


- 在另一个终端中运行 ollama 应用程序或 `ollama serve` 后，在命令行上执行以下命令以试用 80 亿参数的 Llama 3 模型（该模型占用 4.7 GB 存储空间，首次执行此命令时将自动下载）

```bash
# 8B model
ollama run llama3
```


输出如下所示

```
$ ollama run llama3
pulling manifest
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB
pulling 4fa551d4f938... 100% ▕████████████████▏ 12 KB
pulling 8ab4849b038c... 100% ▕████████████████▏ 254 B
pulling 577073ffcc6c... 100% ▕████████████████▏ 110 B
pulling 3f8eb4da87fa... 100% ▕████████████████▏ 485 B
verifying sha256 digest
writing manifest
removing any unused layers
success
```

- 注意 `llama3` 指的是指令微调的 80 亿参数 Llama 3 模型

- 使用 ollama 和 `"llama3"` 模型（80 亿参数模型）需要 16 GB RAM；如果您的机器不支持，可以尝试更小的模型，例如通过设置 `model = "phi-3"` 使用 38 亿参数的 phi-3 模型，仅需 8 GB RAM

- 或者，如果您的机器支持，也可以通过将 `llama3` 替换为 `llama3:70b` 使用更大的 700 亿参数 Llama 3 模型

- 下载完成后，您将看到一个命令行提示，允许您与模型聊天

- 尝试类似 "What do llamas eat?" 的提示，应返回类似以下的输出

```
>>> What do llamas eat?
Llamas are ruminant animals, which means they have a four-chambered
stomach and eat plants that are high in fiber. In the wild, llamas
typically feed on:
1. Grasses: They love to graze on various types of grasses, including tall
grasses, wheat, oats, and barley.
```


- 您可以使用输入 `/bye` 结束此会话


- 以下代码在继续使用 ollama 评估我们在上一节生成的测试集响应之前，检查 ollama 会话是否正常运行


In [ ]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama 未运行。请先启动 ollama 再继续。")
print("Ollama 运行中:", check_if_running("ollama"))


In [ ]:
# 此单元格是可选的；它允许您重启 notebook
# 并仅运行 7.7 节而无需重新运行之前的任何代码
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text


- 现在，与之前用于与模型交互的 `ollama run` 命令不同，可以通过以下函数在 Python 中通过其 REST API 与模型交互
- 在运行此 notebook 的下一个单元格之前，请确保 ollama 仍在运行（之前的代码单元格应打印 `"Ollama 运行中: True"`）
- 接下来，运行以下代码单元格以查询模型


In [ ]:
import requests  # noqa: F811
# import urllib.request

def query_model(
    prompt,
    model="llama3",
    # 如果您使用了 OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # 请将地址从 11434 更新为 11435
    url="http://localhost:11434/api/chat"
):
    # 将数据 payload 创建为字典
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # 以下设置用于确定性响应
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    
    """
    # 将字典转换为 JSON 格式字符串并编码为字节
    payload = json.dumps(data).encode("utf-8")

    # 创建请求对象，设置方法为 POST 并添加必要的 headers
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )
    request.add_header("Content-Type", "application/json")

    # 发送请求并捕获响应
    response_data = ""
    with urllib.request.urlopen(request) as response:
        # 读取并解码响应
        while True:
            line = response.readline().decode("utf-8")
            if not line:
                break
            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data
    """

    # 本书最初使用上面注释掉的代码，基于
    # urllib。它通常工作正常，但一些读者报告
    # 在使用（公司）VPN 时使用 urllib 时出现问题。
    # 下面的代码使用 requests 库，似乎没有
    # 这些问题。

    # 发送 POST 请求
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


model = "llama3"
result = query_model("What do Llamas eat?", model)
print(result)


- 注意，如果您收到 `HTTPError: 404 Client Error: Not Found for url: http://localhost:11434/api/chat` 错误，这可能意味着您尚未下载 `llama3` 模型（要下载模型，请使用 UI 或在终端上运行 `ollama run llama3`）


- 现在，使用上面定义的 `query_model` 函数，我们可以评估微调模型的响应；让我们在上一节查看的前 3 个测试集响应上试一下


In [ ]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


---

**注意：更好的评估提示**

- [一位读者（Ayoosh Kathuria）建议](https://github.com/rasbt/LLMs-from-scratch/discussions/449) 一个更长、改进的提示，在 1–5 分（而非 1 到 100 分）的量表上评估响应，并采用评分标准，从而产生更准确、噪声更少的评估：

```
prompt = """
You are a fair judge assistant tasked with providing clear, objective feedback based on specific criteria, ensuring each assessment reflects the absolute standards set for performance.
You will be given an instruction, a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing the evaluation criteria.
Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
Please do not generate any other opening, closing, and explanations.

Here is the rubric you should use to build your answer:
1: The response fails to address the instructions, providing irrelevant, incorrect, or excessively verbose information that detracts from the user's request.
2: The response partially addresses the instructions but includes significant inaccuracies, irrelevant details, or excessive elaboration that detracts from the main task.
3: The response follows the instructions with some minor inaccuracies or omissions. It is generally relevant and clear, but may include some unnecessary details or could be more concise.
4: The response adheres to the instructions, offering clear, accurate, and relevant information in a concise manner, with only occasional, minor instances of excessive detail or slight lack of clarity.
5: The response fully adheres to the instructions, providing a clear, accurate, and relevant answer in a concise and efficient manner. It addresses all aspects of the request without unnecessary details or elaboration

Provide your feedback as follows:

Feedback:::
Evaluation: (your rationale for the rating, as a text)
Total rating: (your rating, as a number between 1 and 5)

You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

Now here is the instruction, the reference answer, and the response.

Instruction: {instruction}
Reference Answer: {reference}
Answer: {answer}


Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
Feedback:::
Evaluation: """
```

- 更多背景和信息，请参阅[此](https://github.com/rasbt/LLMs-from-scratch/discussions/449) GitHub 讨论

---


- 如我们所见，Llama 3 模型提供了合理的评估，如果模型不完全正确，还会给出部分分数，如 "cumulus cloud" 答案所示
- 注意，之前的提示返回非常冗长的评估；我们可以调整提示以生成 0 到 100 之间的整数响应（100 为最佳），以计算模型的平均分数
- 在 M3 MacBook Air 笔记本电脑上评估测试集中的 110 个条目大约需要 1 分钟


In [ ]:
def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []
    for entry in tqdm(json_data, desc="评分条目"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"无法转换分数: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"分数数量: {len(scores)} / {len(test_data)}")
print(f"平均分数: {sum(scores)/len(scores):.2f}\n")


- 我们的模型平均分数超过 50，可以作为与其他模型比较或尝试其他可能改进模型的训练设置的参考点
- 注意，ollama 在不同操作系统上并非完全确定性（截至本文撰写时），因此您获得的数字可能与上面显示的略有不同


- 供参考，原始
  - Llama 3 8B base 模型得分为 58.51
  - Llama 3 8B instruct 模型得分为 82.65


## 7.9 结论


&nbsp;
### 7.9.1 接下来

- 这标志着本书的最后一章
- 我们涵盖了 LLM 开发周期的主要步骤：实现 LLM 架构、预训练 LLM 并微调

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/21.webp?1" width=500px>

- 如本章所述，指令微调之后有时可选的下一步是偏好微调
- 偏好微调过程对于定制模型以更好地符合特定用户偏好特别有用；如果您感兴趣，请参阅 [../04_preference-tuning-with-dpo](../04_preference-tuning-with-dpo) 文件夹

- 此 GitHub 仓库还包含大量您可能喜欢的额外 bonus 材料；更多信息，请参阅此仓库 README 页面的 [Bonus Material](https://github.com/rasbt/LLMs-from-scratch?tab=readme-ov-file#bonus-material) 部分


&nbsp;
### 7.9.2 在快速发展的领域中保持更新

- 本节无代码


&nbsp;
### 7.9.3 结语

- 希望您喜欢从零实现 LLM 并编写预训练和微调函数的这段旅程
- 在我看来，从零实现 LLM 是理解 LLM 工作原理的最佳方式；希望您通过这种方法有了更深入的理解
- 虽然本书服务于教育目的，但您可能有兴趣在实际应用中使用不同且更强大的 LLM
  - 为此，您可以考虑流行工具，如 axolotl（[https://github.com/OpenAccess-AI-Collective/axolotl](https://github.com/OpenAccess-AI-Collective/axolotl)）或 LitGPT（[https://github.com/Lightning-AI/litgpt](https://github.com/Lightning-AI/litgpt)），我参与开发这些工具


## 总结与要点

- 请参阅 [./gpt_instruction_finetuning.py](./gpt_instruction_finetuning.py) 脚本，这是一个独立的指令微调脚本
- [./ollama_evaluate.py](./ollama_evaluate.py) 是基于 7.8 节的独立脚本，通过 Ollama 和 Llama 3 评估包含 "output" 和 "response" 键的 JSON 文件
- [./load-finetuned-model_ch.ipynb](./load-finetuned-model_ch.ipynb) notebook 演示如何在新会话中加载微调后的模型
- 练习解答见 [./exercise-solutions_ch.ipynb](./exercise-solutions_ch.ipynb)


## 接下来？

- 恭喜完成本书；如果您正在寻找额外资源，我在此 GitHub 仓库中添加了一些 bonus 章节，您可能会感兴趣
- 完整 bonus 材料列表见主 README 的 [Bonus Material](https://github.com/rasbt/LLMs-from-scratch?tab=readme-ov-file#bonus-material) 部分
- 以下是我个人最喜欢的几个：
  1. [Direct Preference Optimization (DPO) for LLM Alignment (From Scratch)](../04_preference-tuning-with-dpo/dpo-from-scratch_ch.ipynb) 实现了一种流行的偏好调优机制，使本章的模型更紧密地符合人类偏好
  2. [Llama 3.2 From Scratch (A Standalone Notebook)](../../ch05/07_gpt_to_llama/standalone-llama32_ch.ipynb)，Meta AI 流行 Llama 3.2 的从零实现，包括加载官方预训练权重；如果您想进行额外实验，可以在各章中将 `GPTModel` 模型替换为 `Llama3Model` 类（它应该可以 1:1 替换）
  3. [Converting GPT to Llama](../../ch05/07_gpt_to_llama) 包含逐步指南代码，解释 GPT-2 与各 Llama 模型之间的差异
  4. [Understanding the Difference Between Embedding Layers and Linear Layers](../../ch02/03_bonus_embedding-vs-matmul/embeddings-and-linear-layers_ch.ipynb) 是一个概念性解释，说明我们在 LLM 输入阶段使用的 PyTorch `Embedding` 层在数学上等价于应用于 one-hot 编码数据的线性层
- 祝您阅读愉快！
